# Middleware in LangChain

Let's start with the core agent loop, then add middleware as hooks around that loop.


In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain langchain-core langchain-openai langgraph python-dotenv

In [ ]:
import getpass
import os

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

In [ ]:
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "middleware"

## The Core Agent Loop

Before middleware matters, lets understand the loop it wraps.

A LangChain agent created with `create_agent()` repeatedly does this:

1. Read the conversation so far.
2. Call the model to decide the next step.
3. If the model asks for a tool, run that tool.
4. Add the tool result back into the conversation.
5. Repeat until the model gives a final answer.

In our support desk analogy, the representative listens to the customer, decides whether to check an internal system, reads the result, and then either checks another system or replies.

### Agent 1: Support Agent With One Tool

This first agent can look up an order status. There is no middleware yet.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a customer's order status by order ID."""
    orders = {
        "A100": "shipped yesterday and arrives tomorrow",
        "B200": "still being packed in the warehouse",
    }
    return orders.get(order_id, "order not found")


support_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[lookup_order],
    system_prompt="You are a concise customer support agent. Use tools when needed.",
)

result = support_agent.invoke(
    {"messages": [{"role": "user", "content": "Can you check order A100?"}]}
)

print(result["messages"][-1].content)

### What Happened?

The user did not directly call `lookup_order`. The agent decided that the tool was useful, called it, read the tool result, and then wrote the final response.

That is the basic agent loop. Middleware becomes useful because real products need control points inside that loop: logging, safety checks, approvals, retries, and clean error handling.

## 3. Where Middleware Hooks Fit

Middleware is code that runs around important moments in the loop. It does not replace the agent. It wraps pieces of the agent's work.

Using the support desk analogy:

- Before the representative talks to the customer, the shift lead can add policy reminders.
- Before the representative uses an internal system, the shift lead can check whether that action is allowed.
- After an internal system responds, the shift lead can log what happened.
- If an internal system fails, the shift lead can recover with a helpful fallback.

LangChain exposes middleware hooks around the model call, tool call, and agent lifecycle. The official middleware overview summarizes this as hooks before and after core agent-loop steps.

### Tiny Agent 2: Observe Tool Calls With Middleware

This example adds middleware that logs every tool call. It is intentionally small: the middleware does not change behavior yet; it only observes.

The shift lead is now watching which internal system the representative uses.

In [ ]:
from langchain.agents.middleware import wrap_tool_call


@wrap_tool_call
def log_tool_call(request, handler):
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    print(f"[middleware] about to call tool: {tool_name} with {tool_args}")

    response = handler(request)

    print(f"[middleware] finished tool: {tool_name}")
    return response


support_agent_with_logging = create_agent(
    model="openai:gpt-4o-mini",
    tools=[lookup_order],
    middleware=[log_tool_call],
    system_prompt="You are a concise customer support agent. Use tools when needed.",
)

result = support_agent_with_logging.invoke(
    {"messages": [{"role": "user", "content": "What is happening with order B200?"}]}
)

print(result["messages"][-1].content)

### Teaching Notes

The important idea is not the `print()` statements. The important idea is the shape:

```python
@wrap_tool_call
def some_middleware(request, handler):
    # before the tool runs
    response = handler(request)
    # after the tool runs
    return response
```

`request` is the current thing the agent is trying to do. `handler(request)` continues the normal agent behavior. Anything before or after `handler(request)` is your hook.

That is middleware: code inserted into the agent loop at a defined control point.